<a href="https://colab.research.google.com/github/Inc-pixel/Metal-Subgenres/blob/main/Stage1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from itertools import combinations
%pip install seaborn
%pip install pandas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, integrate
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.multitest import multipletests

os.makedirs("figures", exist_ok=True)   # folder for saved charts
pd.set_option("display.width", 160)     # wider printed tables

In [2]:
import os
from google.colab import drive
drive.mount("/content/drive")
# os.chdir("/content/drive/MyDrive/heavier-than-thou")   # your project folder

Mounted at /content/drive


In [3]:
df = pd.read_csv("dataset.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")   # leftover row numbers

print(df.shape)   # (rows, columns)
df.head()         # first 5 rows

FileNotFoundError: [Errno 2] No such file or directory: 'dataset.csv'

In [ ]:
print(df["track_genre"].nunique(), "genre labels")
print(df["track_genre"].value_counts().head(100))



In [ ]:
metalish = sorted(g for g in df["track_genre"].unique() if isinstance(g, str) and ("metal" in g or "core" in g))
print(metalish)

In [ ]:
GENRES = ["metalcore", "death-metal", "black-metal", "heavy-metal", "grindcore"]
FEATURES = ["energy", "loudness", "valence", "danceability", "tempo", "instrumentalness"]

funnel = []   # remembers how many rows survive each step

def log_step(step, data):
    funnel.append({"step": step, "rows": len(data)})
    print(f"{step:<42} {len(data):>5} rows")

In [ ]:
metal = df[df["track_genre"].isin(GENRES)].copy()
log_step("1. Keep the 5 metal subgenres", metal)

In [ ]:

metal = metal.drop_duplicates(subset=["track_id", "track_genre"])
log_step("2. Drop repeated track IDs", metal)

In [ ]:
labels_per_track = metal.groupby("track_id")["track_genre"].nunique()
multi_label = labels_per_track[labels_per_track > 1].index
print("Tracks with 2+ subgenre labels:", len(multi_label))

metal = metal[~metal["track_id"].isin(multi_label)]   # ~ means "not"
log_step("3. Drop tracks with 2+ subgenre labels", metal)

In [ ]:
metal = metal.drop_duplicates(subset=["artists", "track_name", "track_genre"])
log_step("4. Drop re-releases of the same song", metal)

In [ ]:
CAP = 10
metal = (metal.sample(frac=1, random_state=42).groupby(["track_genre", "artists"])
         .head(CAP)) # keep up to CAP per band

log_step(f"7. Cap each artist at {CAP} tracks", metal)
print(metal["track_genre"].value_counts())


In [ ]:
for feat in FEATURES:
    by_genre = metal.groupby("track_genre")[feat]
    q1 = by_genre.transform(lambda s: s.quantile(0.25))
    q3 = by_genre.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    flagged = (metal[feat] < q1 - 1.5 * iqr) | (metal[feat] > q3 + 1.5 * iqr)
    print(f"{feat: <18} {flagged.sum() >4} flagged")

In [ ]:
metal.to_csv("metal_clean.csv", index=False)
pd.DataFrame(funnel).to_csv("cleaning_funnel.csv", index=False)
pd.DataFrame(funnel)